In [ ]:
%reload_ext autoreload
# %load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from tqdm import tqdm
import itertools
from g_anomaly import PA_with_anomaly_numba
from g_standard import standard_PA_numba
from support_tools import (
    edge_degree_counting,
    edge_degree_counting_many,
    expected_degree_PA_normal_all_vertices,
    graph_series_connected_vertex_degree,
    get_top_vertices,
    prepare_screening,
    get_vertex_ranks
)

from estimation_pa_anomaly_fixed import single_estimation_beta
from estimation_pa_normal import optimize_parameters_normal
from pa_loglikelihood import neg_log_likelihood_normal, neg_log_likelihood_anomaly
from collections import defaultdict

In [ ]:
################ detection for known delta ##############

def run_single_dataset_estimate_fixed_delta_known(beta_bounds, delta_fixed, network_size, m, edge_list, k_result, vertex_list):
    """
    Assume delta is known
    procedure: 
        1. estimation: find several candidates, find beta given fixed delta
        2. find the vertex that has maximum log likelihood value
    return:
        tau_hat, beta_hat, log-likelihood value
    """
    
    beta_list, delta_list, log_L_list= [], [], []
        
    edges_np = np.asarray(edge_list, dtype=np.int64)          # shape (n,2)
    verts_np = np.asarray(vertex_list, dtype=np.int64)        # 0-based candidates

    deg_all, recv_all = edge_degree_counting_many(verts_np, edges_np, network_size)

    for i in range(len(vertex_list)):
        vertex_i = vertex_list[i] + 1  # likelihood: 1-based
        degree_series_i = deg_all[i].astype(np.float64)   
        edge_adj_i      = recv_all[i].astype(np.float64) 
        # delta is known 
        beta_i = single_estimation_beta(beta_bounds, delta_fixed, network_size, vertex_i, k_result, degree_series_i, edge_adj_i,m)
        log_L_i = neg_log_likelihood_anomaly(beta_i, delta_fixed, network_size, vertex_i,
                                        k_result, degree_series_i,edge_adj_i,m)
        log_L_list.append(log_L_i)
        beta_list.append(beta_i)
            #print(beta_i, delta_i, log_L_i)
    vertex_best = np.argmin(log_L_list)
    tau_hat = vertex_list[vertex_best]
    beta_hat = round(beta_list[vertex_best],4)
    log_L_hat = round(log_L_list[vertex_best],4)

    return tau_hat, beta_hat, log_L_hat

In [ ]:
# --------- standard PA data, to find the critical value ---------

def get_critical_value_known_delta(rep_num, seed_num, network_size, m, delta, exp_degree, top_k_degree, top_k_ratio, delta_r, beta_bound):
    log_lambda_0 = []
    for rep_2 in tqdm(range(rep_num)):
        seed_2 = seed_num + 123 + rep_2

        edges_2 = standard_PA_numba(network_size, m, delta, seed=seed_2)

        k_series_2 = np.asarray(graph_series_connected_vertex_degree(edges_2, receiver_index=1), dtype=np.float64)
    
        prepared_2 = prepare_screening(edges_2, exp_degree, Top_degree_rank, Top_ratio_rank, delta_r, store_ratio=True)
        vertex_list_2 = prepared_2["vertex_list"]

        tau_2_e, beta_2_e, nll_e = run_single_dataset_estimate_fixed_delta_known(beta_bound, delta, 
                                                                 network_size, m, edges_2, k_series_2, vertex_list_2)

        nll0_2 = neg_log_likelihood_normal(delta, network_size, k_series_2, m)
        
        lambda_e = -nll_e + nll0_2
        log_lambda_0.append(lambda_e)
    
    critical_value = np.percentile(log_lambda_0, 95)
    return critical_value

In [ ]:
def main_detection_procedure_delta_known(rep_num, seed_num, network_size, m, beta, delta, vertex_tau, top_k_degree, top_k_ratio, beta_bound):
    """
    The procedure to detection the anomaly with known delta:
    1. Since we known delta, the first step we can find the critical value by simulation
    2. Find the log-likelihood ratio of the network and compare it with the critical value
    """
    delta_r = 0    # for calculating the nominal ratio
    # find critical value
    exp_degree = expected_degree_PA_normal_all_vertices(network_size, delta_r, m)
    c_0 = get_critical_value_known_delta(rep_num, seed_num, network_size, m, delta, exp_degree, top_k_degree, top_k_ratio, delta_r, beta_bound)
    results = []

    # --------- Under alternative (anomaly data) ---------
    for rep_1 in tqdm(range(rep_num)):
        seed_1 = base_seed + rep_1

        edges_1 = PA_with_anomaly_numba(network_size, m, beta, vertex_tau, delta, seed_1)

        k_series_1 = np.asarray(graph_series_connected_vertex_degree(edges_1, receiver_index=1), dtype=np.float64)
        prepared_1 = prepare_screening(edges_1, exp_degree, Top_degree_rank, Top_ratio_rank, delta_r, store_ratio=True)
        vertex_list_1 = prepared_1["vertex_list"]

        # IMPORTANT: pass vertex_tau if estimator needs it
        tau_1, beta_1, nll_1 = run_single_dataset_estimate_fixed_delta_known(beta_bound, delta, 
                                                                 network_size, m, edges_1, k_series_1, vertex_list_1)
        tau_d_rank, tau_r_rank = get_vertex_ranks(tau_1, prepared_1,delta_r)

        
        nll0_1 = neg_log_likelihood_normal(delta_true, network_size, k_series_1, m)
        
        lambda_1 = - nll_1 + nll0_1

        if lambda_1 > c_0:
            detection_res = "Reject H_0"
        else:
            detection_res = "Accept H_0"
        rec = {
            "replicate": rep_1,
            "seed": seed_1,
            "beta_true": beta,
            "delta_true": delta,
            "tau_true": vertex_tau,
            "beta_hat": float(beta_1),
            "tau_hat": tau_1,
            "degree_rank": tau_d_rank,
            "ratio_rank": tau_r_rank,
            "nll1": float(nll_1),
            "nll0": float(nll0_1),
            "lambda": float(lambda_1),
            "critical_value": c_0,
            "detection_result": detection_res
        }
        results.append(rec)
    return results

In [ ]:
def main():
    # sample parameters used for this run
    seed_num_tmp = 12345
    network_size_tmp = 10000
    m_tmp = 3
    delta_true_tmp = 0.1
    beta_true_tmp = 0.005
    vertex_tau_tmp = 5000

    top_degree_k_tmp = 5
    top_ratio_k_tmp = 5
    rep_num_tmp = 5

    beta_bound_tmp = [0.000001, 10]

    result_details = main_detection_procedure_delta_known(
        rep_num_tmp,
        seed_num_tmp,
        network_size_tmp,
        m_tmp,
        beta_true_tmp,
        delta_true_tmp,
        vertex_tau_tmp,
        top_degree_k_tmp,
        top_ratio_k_tmp,
        beta_bound_tmp
    )

    print(result_details)


if __name__ == "__main__":
    main()